<a href="https://colab.research.google.com/github/notmg7/Northstar-analytics/blob/main/SQL_in_R.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **SQL in R: CRUD Operations**

**Installing and Loading Required Packages**

In [ ]:
# Installing and Loading required packages
install.packages(c("tidyverse", "sqldf", "RCurl"))
library(tidyverse)
library(sqldf)
library(RCurl)

Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘gsubfn’, ‘proto’, ‘RSQLite’, ‘chron’, ‘bitops’


── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.3     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Loading required package: gsubfn

Loading required package: proto

Warning message:
“no DISPLAY variable so Tk is not available”
Loading required package: RSQLite


Attaching package: ‘RCurl’


The following object is masked from ‘package:tidyr’:

    complete




**Importing and Verifying Datasets**

In [ ]:
# Importing Data from GitHub
orders <- read_csv("https://github.com/notmg7/Northstar-analytics/raw/refs/heads/main/northstar_dataset/orders.csv")
deliveries <- read_csv("https://github.com/notmg7/Northstar-analytics/raw/refs/heads/main/northstar_dataset/deliveries.csv")
hubs <- read_csv("https://github.com/notmg7/Northstar-analytics/raw/refs/heads/main/northstar_dataset/hubs.csv")

# Verifying Data
glimpse(orders)
glimpse(deliveries)
glimpse(hubs)

Rows: 1250 Columns: 11
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (7): order_id, customer_id, service_type, pickup_zone, dropoff_zone, pr...
dbl  (3): promised_window_hours, order_value, special_handling_flag
dttm (1): order_created_at

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 950 Columns: 13
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (6): delivery_id, order_id, driver_id, vehicle_id, hub_id, delivery_status
dbl  (5): route_distance_km, manual_route_override_count, proof_of_completio...
dttm (2): dispatch_time, delivery_completed_at

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 8 Columns: 5
── Column specification ────────────────────────────

Rows: 1,250
Columns: 11
$ order_id              <chr> "O00001", "O00002", "O00003", "O00004", "O00005"…
$ customer_id           <chr> "C0292", "C0459", "C0161", "C0520", "C0558", "C0…
$ service_type          <chr> "Passenger", "Passenger", "Passenger", "Parcel",…
$ order_created_at      <dttm> 2024-08-20 14:43:00, 2024-05-14 22:16:00, 2025-…
$ promised_window_hours <dbl> 6, 24, 4, 2, 12, 1, 2, 4, 12, 6, 12, 12, 24, 6, …
$ pickup_zone           <chr> "Airport", "North", "West", "RiverSide", "Rivers…
$ dropoff_zone          <chr> "South", "AIRPORT", "AIRPORT", "North", "SOUTH",…
$ priority_level        <chr> "Medium", "Low", "High", "Medium", "Low", "High"…
$ order_value           <dbl> 126.65, 109.30, 33.50, 10.04, 125.58, 151.44, 76…
$ booking_channel       <chr> "App", "App", "Phone", "App", "Phone", "Web", "A…
$ special_handling_flag <dbl> 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, …
Rows: 950
Columns: 13
$ delivery_id                   <chr> "DL00001", "DL00002", "DL00003", "DL

Select **Operations**

In [ ]:
# Query 1: Find high-priority failures by joining Orders and Deliveries
q1 <- sqldf("
  SELECT o.order_id, o.service_type, o.priority_level, d.delivery_status
  FROM orders o
  JOIN deliveries d ON o.order_id = d.order_id
  WHERE o.priority_level = 'High'
    AND d.delivery_status = 'Failed'
  LIMIT 10
")
print(q1)

   order_id service_type priority_level delivery_status
1    O00084    Passenger           High          Failed
2    O00089      Medical           High          Failed
3    O00113     Business           High          Failed
4    O00123       Retail           High          Failed
5    O00129       Retail           High          Failed
6    O00142       Retail           High          Failed
7    O00158     Business           High          Failed
8    O00192       Retail           High          Failed
9    O00203     Business           High          Failed
10   O00204    Passenger           High          Failed


In [ ]:
# Query 2: Calculate total deliveries per hub zone
q2 <- sqldf("
  SELECT h.zone, h.hub_name, COUNT(d.delivery_id) AS total_jobs
  FROM hubs h
  JOIN deliveries d ON h.hub_id = d.hub_id
  GROUP BY h.zone
  ORDER BY total_jobs DESC
")
print(q2)

       zone       hub_name total_jobs
1   Central   Central Core        243
2     North North Exchange        136
3      West      West Gate        127
4      East      East Dock        119
5 Riverside  Riverside Hub        115
6     South     South Link        106
7   Airport    Airport Hub        104


**Create Operation**

In [ ]:
sqldf()

# CREATE the table
sqldf("
  CREATE TABLE system_audit_log (
    log_id INTEGER PRIMARY KEY,
    event_type TEXT,
    event_description TEXT,
    status TEXT
  )
")

# Verify it exists in this session
sqldf("SELECT name FROM sqlite_master WHERE type='table' AND name='system_audit_log'")

<SQLiteConnection>
  Path: :memory:
  Extensions: TRUE

Warning message in result_fetch(res@ptr, n = n):
“`dbGetQuery()`, `dbSendQuery()` and `dbFetch()` should only be used with `SELECT` queries. Did you mean `dbExecute()`, `dbSendStatement()` or `dbGetRowsAffected()`?”


<0 x 0 matrix>

name
<chr>
system_audit_log


**Insert Operations**

In [ ]:
# INSERT Operation
hubs_updated <- sqldf(c(
  "INSERT INTO hubs VALUES ('H09', 'Coastal Point', 'East', 'Dispatch', 75)",
  "SELECT * FROM main.hubs"
))

# Verify the insertion
tail(hubs_updated, 3)

Warning message in result_fetch(res@ptr, n = n):
“`dbGetQuery()`, `dbSendQuery()` and `dbFetch()` should only be used with `SELECT` queries. Did you mean `dbExecute()`, `dbSendStatement()` or `dbGetRowsAffected()`?”


,hub_id,hub_name,zone,hub_type,capacity_score
,<chr>,<chr>,<chr>,<chr>,<dbl>
7,H07,Riverside Hub,Riverside,Warehouse,66
8,H08,Midtown Relay,Central,Charging,63
9,H09,Coastal Point,East,Dispatch,75


In [ ]:
# INSERT data into the existing table
sqldf("
  INSERT INTO system_audit_log (log_id, event_type, event_description, status)
  VALUES
  (1, 'System_Check', 'Initial verification of the audit table.', 'Success'),
  (2, 'System_Check', 'Testing multi-row insertion capability.', 'Active')
")

# SELECT to show it worked for your report
verification_view <- sqldf("SELECT * FROM system_audit_log")
print(verification_view)

Warning message in result_fetch(res@ptr, n = n):
“`dbGetQuery()`, `dbSendQuery()` and `dbFetch()` should only be used with `SELECT` queries. Did you mean `dbExecute()`, `dbSendStatement()` or `dbGetRowsAffected()`?”


<0 x 0 matrix>

  log_id   event_type                        event_description  status
1      1 System_Check Initial verification of the audit table. Success
2      2 System_Check  Testing multi-row insertion capability.  Active


**Delete Operation**

In [ ]:
# 1. Audit: Count verification records before deletion
sqldf("SELECT COUNT(*) AS records_to_purge FROM system_audit_log WHERE event_type = 'System_Check'")

# 2. DELETE: Purge the sample records from the audit log
sqldf("DELETE FROM system_audit_log WHERE event_type = 'System_Check'")

# 3. Verify: Confirm the table has been cleaned
cleaned_audit <- sqldf("SELECT * FROM system_audit_log")
print(cleaned_audit)

sqldf()

records_to_purge
<int>
2


Warning message in result_fetch(res@ptr, n = n):
“`dbGetQuery()`, `dbSendQuery()` and `dbFetch()` should only be used with `SELECT` queries. Did you mean `dbExecute()`, `dbSendStatement()` or `dbGetRowsAffected()`?”


<0 x 0 matrix>

[1] log_id            event_type        event_description status           
<0 rows> (or 0-length row.names)


NULL

**Update Operation**

In [ ]:
# Audit: Identify deliveries with high overrides marked as OnTime
sqldf("SELECT COUNT(*) FROM deliveries WHERE manual_route_override_count > 2 AND delivery_status = 'OnTime'")

# UPDATE: Reclassify these records using SQL syntax
deliveries_cleaned <- sqldf(c(
  "UPDATE deliveries
   SET delivery_status = 'Manual-High'
   WHERE manual_route_override_count > 2
   AND delivery_status = 'OnTime'",
  "SELECT * FROM deliveries"
))

# Verify: Confirm the new status exists
sqldf("SELECT delivery_status, COUNT(*) FROM deliveries_cleaned GROUP BY delivery_status")

COUNT(*)
<int>
52


Warning message in result_fetch(res@ptr, n = n):
“`dbGetQuery()`, `dbSendQuery()` and `dbFetch()` should only be used with `SELECT` queries. Did you mean `dbExecute()`, `dbSendStatement()` or `dbGetRowsAffected()`?”


delivery_status,COUNT(*)
<chr>,<int>
Delayed,202
Failed,132
Manual-High,52
OnTime,564


**Aggregate Functions**

In [ ]:
# SQL Mathematical and Aggregate Functions: Financial Performance by Zone
kpi_summary <- sqldf("
  SELECT
    h.zone,
    COUNT(d.delivery_id)                AS total_deliveries,
    ROUND(SUM(o.order_value), 2)        AS total_revenue,
    ROUND(AVG(o.order_value), 2)        AS avg_order_value,
    ROUND(SUM(d.fuel_or_charge_cost), 2) AS energy_costs,
    ROUND(SUM(o.order_value) -
          SUM(d.fuel_or_charge_cost), 2) AS net_profit,
    ROUND(100.0 * (SUM(o.order_value) - SUM(d.fuel_or_charge_cost)) /
          NULLIF(SUM(o.order_value), 0), 1) AS profit_margin_pct
  FROM deliveries d
  JOIN orders o ON d.order_id = o.order_id
  JOIN hubs h   ON d.hub_id = h.hub_id
  GROUP BY h.zone
  ORDER BY net_profit DESC
")

print(kpi_summary)

       zone total_deliveries total_revenue avg_order_value energy_costs
1   Central              243      22828.80           93.95      3072.54
2     North              136      12478.45           91.75      1734.79
3      East              119      11624.51           97.68      1516.56
4      West              127      11438.18           90.06      1672.21
5 Riverside              115      10342.23           89.93      1486.04
6     South              106       9577.61           90.35      1331.89
7   Airport              104       8977.99           86.33      1385.20
  net_profit profit_margin_pct
1   19756.26              86.5
2   10743.66              86.1
3   10107.95              87.0
4    9765.97              85.4
5    8856.19              85.6
6    8245.72              86.1
7    7592.79              84.6


# **R Analytics and Visualization**

**Environment Setup & Data Integration**

In [ ]:
# Load specialized libraries
install.packages(c("e1071", "RColorBrewer"))
library(e1071)
library(RColorBrewer)

# 2. Create the Integrated Analytics View
# Joining core tables for a unified operational dataset
master_analytics <- deliveries %>%
  left_join(orders, by = "order_id") %>%
  left_join(hubs, by = "hub_id")

# Verify the join
glimpse(master_analytics)

Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependency ‘proxy’



Attaching package: ‘e1071’


The following object is masked from ‘package:ggplot2’:

    element




Rows: 950
Columns: 27
$ delivery_id                   <chr> "DL00001", "DL00002", "DL00003", "DL0000…
$ order_id                      <chr> "O00938", "O00004", "O00639", "O00313", …
$ driver_id                     <chr> "D004", "D138", "D006", "D116", "D108", …
$ vehicle_id                    <chr> "V056", "V007", "V049", "V055", "V034", …
$ hub_id                        <chr> "H05", "H02", "H02", "H02", "H01", "H03"…
$ dispatch_time                 <dttm> 2024-06-18 10:57:00, 2025-01-11 18:45:0…
$ delivery_completed_at         <dttm> 2024-06-19 09:05:59, 2025-01-11 17:39:0…
$ delivery_status               <chr> "Failed", "OnTime", "OnTime", "Delayed",…
$ route_distance_km             <dbl> 17.26, 10.34, 7.92, 16.42, 14.52, 13.84,…
$ manual_route_override_count   <dbl> 1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 0, 3, 2, 0…
$ proof_of_completion_missing   <dbl> 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0…
$ customer_rating_post_delivery <dbl> 3.07, 5.00, 4.98, 4.18, 4.18, 1.57, 4.64…
$ fuel_or_charge_c

**Statistical Analysis**

In [ ]:
# Environment & Data Preparation
install.packages("effsize")
library(effsize); library(tidyverse)

master_analytics <- deliveries %>%
  left_join(orders, by = "order_id") %>%
  mutate(delay_mins = as.numeric(difftime(as.POSIXct(delivery_completed_at),
                                         as.POSIXct(dispatch_time), units = "mins"))) %>%
  filter(!is.na(delay_mins))

# Descriptive & Inferential Analysis
print(summary(master_analytics$delay_mins))
cat('Skewness: ', e1071::skewness(master_analytics$delay_mins, na.rm=TRUE), '\n')

group_a <- master_analytics$delay_mins[master_analytics$service_type == "Business"]
group_b <- master_analytics$delay_mins[master_analytics$service_type == "Medical"]

t.test(group_a, group_b)
cohen.d(group_a, group_b, na.rm=TRUE)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
 -132.9   176.8   424.5   572.7   878.6  2607.4 
Skewness:  1.028192 



	Welch Two Sample t-test

data:  group_a and group_b
t = -0.67143, df = 204.56, p-value = 0.5027
alternative hypothesis: true difference in means is not equal to 0
95 percent confidence interval:
 -190.15360   93.54175
sample estimates:
mean of x mean of y 
 550.3656  598.6715 



Cohen's d

d estimate: -0.09029247 (negligible)
95 percent confidence interval:
     lower      upper 
-0.3517503  0.1711653 


**Visualisation 1 - Delay Distribution (Box Plot)**

In [ ]:
# Creating the Box Plot to visualize regional performance
ggplot(master_analytics, aes(x = delay_mins, y = reorder(zone, delay_mins, FUN = median, na.rm = TRUE), fill = zone)) +
  geom_boxplot(outlier.colour = "red", outlier.shape = 16, outlier.size = 2) +
  scale_fill_brewer(palette = "Blues", direction = -1) +
  labs(title = "Regional Delivery Delay Distribution",
       subtitle = "Red dots indicate extreme delivery failures (outliers)",
       x = "Delay (Minutes)",
       y = "Operational Zone") +
  theme_minimal() +
  theme(legend.position = "none")

**Visualization 2 - Exception Rate by Service Type (Bar Chart)**

In [ ]:
# Calculate Exception Rates by Service Type
exception_summary <- master_analytics %>%
  group_by(service_type) %>%
  summarise(
    total = n(),
    exceptions = sum(delivery_status %in% c('Failed', 'Delayed', 'Manual-High')),
    exception_rate = round(100 * exceptions / total, 1)
  )

# Generate the Bar Chart
ggplot(exception_summary, aes(x = reorder(service_type, exception_rate), y = exception_rate, fill = exception_rate)) +
  geom_col(width = 0.65) +
  geom_text(aes(label = paste0(exception_rate, '%')), hjust = -0.2, fontface = 'bold', size = 4) +
  coord_flip() +
  scale_fill_gradient(low = '#BDD7EE', high = '#C55A11') +
  labs(
    title = 'Operational Exception Rate by Service Type',
    subtitle = 'Higher rates indicate systemic failure in service delivery',
    x = 'Service Type',
    y = 'Exception Rate (%)',
    caption = 'Source: NorthStar Operational Dataset'
  ) +
  theme_minimal(base_size = 13) +
  theme(
    legend.position = 'none',
    plot.title = element_text(face='bold', colour='#1F4E79')
  )

**Visualization 3 - Cost vs Delay Scatter Plot**

In [ ]:
# Visualisation 3: Scatter plot — cost vs delay with linear regression
ggplot(master_analytics, aes(x = delay_mins, y = fuel_or_charge_cost, colour = zone)) +
  geom_point(alpha = 0.45, size = 1.8) +
  geom_smooth(method = 'lm', se = TRUE, colour = '#C55A11', fill = '#FAD7A0', linewidth = 1.2) +
  facet_wrap(~ service_type, scales = 'free') +
  scale_colour_brewer(palette = 'Set1') +
  labs(
    title    = 'Journey Cost vs Delay — Faceted by Service Type',
    subtitle = 'Flat trend lines indicate independence between cost and performance',
    x = 'Delay (minutes)',
    y = 'Fuel/Charge Cost (GBP)',
    colour = 'Zone',
    caption = 'Source: NorthStar Operational Dataset'
  ) +
  theme_bw(base_size = 12) +
  theme(
    plot.title = element_text(face='bold', colour='#1F4E79'),
    strip.background = element_rect(fill='#1F4E79'),
    strip.text = element_text(colour='white', face='bold')
  )